In [ ]:
import os

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from wsd.models import JMDict, PointWiseRanker, GeminiRanker
from wsd.utils import load_dataset, accuracy

In [ ]:
basedir = os.getenv('PJ_DIR')
X, y = load_dataset(f'{basedir}/data/dataset_.xml')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [ ]:
jmdict_base = JMDict()
base_preds = jmdict_base.predict(X_test)

In [ ]:
model = LogisticRegression(C=10, penalty='l1', solver='liblinear')
ranker = PointWiseRanker(ranking_model=model)
jmdict_basic = JMDict(ranker=ranker)
jmdict_basic.fit(X_train, y_train)
basic_preds = jmdict_basic.predict(X_test)

In [ ]:
ranker = GeminiRanker("gemini-2.5-flash-lite")
jmdict_gemini = JMDict(ranker=ranker)
gemini_preds = jmdict_gemini.predict(X_test)

In [ ]:
print(f"baseline = {accuracy(base_preds, y_test):.2%}")
print(f"basic\t = {accuracy(basic_preds, y_test):.2%}")
print(f"gemini\t = {accuracy(gemini_preds, y_test):.2%}")

In [ ]:
import pandas as pd
import numpy as np

jmdict_basic.fit(X, y)
pd.DataFrame(
    jmdict_basic.ranker.model.coef_, 
    columns=jmdict_basic.ranker.vec.get_feature_names_out()
).T\
    .sort_values(by=0, ascending=False)\
    .plot(kind='barh', figsize=(5, 14), title='Basic Ranker Coefficients')

In [ ]:
jmdict_basic.ranker.save('../data/jmdict_basic_ranker.joblib')